# 📘 부록: Hugging Face 허브에 임베딩 모델과 데이터셋 올리기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">허브 공유</strong>는 학습한 모델과 사용한 데이터를 내 Hugging Face 저장소에 올리고, 저장소 이름으로 다른 환경에서 다시 받는 일입니다. 모델 저장소와 데이터셋 저장소를 나누면 학습 결과와 데이터의 용도를 구분해 재사용할 수 있습니다.</p></div>

<img src="images/50_embedding_hub.png" width="1050" alt="모델 전체와 학습·검증 쌍·검색 문서를 허브에 올리고 다른 파드에서 다시 받습니다.">

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01에서 저장한 노무 임베딩 모델과 사용한 데이터를 허브에 올립니다. 노무 학습·검증 자료와 함께, 다른 분야의 검색 퇴행을 확인하는 일반 평가 쌍·문서도 공유합니다. 마지막에는 허브에서 받은 모델과 문서로 새 질의를 검색합니다.</p></div>

## 1. 공유할 결과와 쓰기 토큰

### 1-1. 모델과 데이터셋

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01의 모델 저장까지 마친 RunPod 파드에서 새 커널로 시작합니다. 모델 저장소에는 <code style="font-size:0.95em;color:#345473;">output/labor_tuned</code> 전체를, 데이터셋 저장소에는 노무 학습·검증·검색 문서와 일반 검색 퇴행 평가 자료를 올립니다.</p></div>

데이터셋은 day71 부록과 같은 `load_dataset` → `push_to_hub` 방식입니다. 모델은 `ModelCard` → `create_repo` → `upload_folder` 순서로 올리며, 이번에는 어댑터 대신 전체 임베딩 모델 폴더를 사용합니다.

### 1-2. 쓰기 토큰

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">day71 부록에서 만든 쓰기 토큰을 그대로 사용합니다. 처음 만들면 아래 화면에서 Write 권한을 선택하고, 받은 값을 단원 폴더 <code style="font-size:0.95em;color:#345473;">.env</code>에 <code style="font-size:0.95em;color:#345473;">HF_TOKEN=복사한_토큰</code>으로 적습니다.</p></div>

- 누를 곳: 프로필 사진 → Settings → Access Tokens → Create new token → Write
- 주소: [쓰기 토큰 만들기](https://huggingface.co/settings/tokens/new?tokenType=write)

이 부록은 기존 부록처럼 공개 저장소로 올립니다. 비공개로 올릴 때는 데이터셋의 `push_to_hub`와 모델의 `create_repo`에서 모두 `private=True`로 바꿉니다. 이미 있는 저장소의 공개 범위는 저장소 Settings에서 확인합니다. 토큰 값은 출력하거나 올리지 않습니다.

## 2. 실습 준비와 저장소 이름

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">day71 부록과 같은 코드로 <code style="font-size:0.95em;color:#345473;">.env</code>를 읽고 토큰의 계정·권한을 확인합니다. 단원 <code style="font-size:0.95em;color:#345473;">.env</code>를 먼저 읽고, 상위 실습자료 <code style="font-size:0.95em;color:#345473;">.env</code>에서 없는 값만 채웁니다.</p></div>

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import HfApi

material_dir = Path(".")  # data가 있는 단원 폴더
model_dir = material_dir / "output" / "labor_tuned"  # 교안 01에서 저장한 전체 임베딩 모델

# 단원 폴더의 .env를 먼저 읽고, 없는 키만 상위 실습자료 폴더의 .env에서 채웁니다.
load_dotenv(material_dir / ".env")
load_dotenv(material_dir.resolve().parent / ".env")
hf_token = os.getenv("HF_TOKEN")

# 토큰의 계정과 권한을 봅니다. 올리려면 쓰기 권한이 있어야 합니다.
api = HfApi(token=hf_token)
account = api.whoami()
user_name = account["name"]
print("계정:", user_name, "| 권한:", account["auth"]["accessToken"]["role"])

# 계정 이름 뒤에 데이터셋과 모델 저장소 이름을 붙입니다.
dataset_repo_id = f"{user_name}/labor-embedding-data"
model_repo_id = f"{user_name}/multilingual-e5-small-labor"
print(dataset_repo_id, "|", model_repo_id)

## 3. 데이터셋 올리기

### 3-1. JSONL을 DatasetDict로 읽기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">이 실습에서는 <strong style="font-size:1.13em;color:#1f3a5f;">조문 하나가 청크 역할</strong>을 하며, <code style="font-size:0.95em;color:#345473;">corpus</code>는 그 조문들을 모은 <strong style="font-size:1.13em;color:#1f3a5f;">검색 대상 문서 모음</strong>입니다. <code style="font-size:0.95em;color:#345473;">train</code>은 질문·정답 조문으로 모델의 가중치를 학습하고, <code style="font-size:0.95em;color:#345473;">eval</code>은 학습에 넣지 않은 질문·정답 조문 ID로 학습 전후 검색 성능을 평가합니다.</p></div>

day71처럼 `load_dataset("json", data_files=...)`로 학습·검증 파일을 같은 `DatasetDict`의 `train`·`validation`으로 읽습니다. 검색 문서는 열이 달라 별도의 **config**(용도별 데이터 묶음)로 올립니다. 원문에는 `query: `와 `passage: `를 덧붙이지 않습니다.

| config 이름 | 용도 | 원본 파일 | split | 주요 열 |
|---|---|---|---|---|
| `training-pairs` | 질문·정답 조문으로 모델 학습 | `labor_train.jsonl` | `train` | `anchor`, `positive`, `doc_id` |
| `training-pairs` | 학습 전후 검색 성능 평가 | `labor_eval.jsonl` | `validation` | `anchor`, `positive`, `doc_id` |
| `corpus` | 검색할 조문 모음 | `labor_corpus.jsonl` | `corpus` | `doc_id`, `law`, `topic`, `text` |
| `general-evaluation` | 다른 분야 검색의 퇴행 점검(83쌍) | `general_eval.jsonl` | `test` | `anchor`, `positive`, `doc_id` |
| `general-corpus` | 일반 평가의 검색 문서(433개) | `general_corpus.jsonl` | `corpus` | `doc_id`, `domain`, `title`, `text` |

학습·검증은 같은 세 열로 저장하며, 검색 평가에는 검증 쌍의 `anchor`와 `doc_id`를 사용합니다. 원조문에서 질문·정답 쌍을 만드는 방법은 교안 01의 데이터 구성 절에서 확인합니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">노무 검증</strong>은 같은 분야의 새 질문으로 일반화를, <strong style="font-size:1.13em;color:#1f3a5f;">일반 평가</strong>는 관광·상식·가상 도서관 검색의 변화를 확인합니다. 일반 평가 자료는 모델 학습에 넣지 않습니다. 평가할 때는 정답 ID로 문서의 <code style="font-size:0.95em;color:#345473;">domain</code>을 확인하고 각 분야의 후보 문서에서 검색해야 교안과 같은 조건입니다.</p></div>

In [ ]:
from datasets import load_dataset

# 노무 학습·검증 쌍을 함께 읽고, 열 구조가 다른 검색 문서는 나눠 읽습니다.
labor_dataset = load_dataset("json", data_files={
    "train": str(material_dir / "data" / "본실습" / "labor_train.jsonl"),
    "validation": str(material_dir / "data" / "본실습" / "labor_eval.jsonl"),
})
labor_corpus_dataset = load_dataset("json", data_files={
    "corpus": str(material_dir / "data" / "본실습" / "labor_corpus.jsonl"),
})
# 일반 검색 퇴행 평가용이며, 모델의 가중치 학습에는 넣지 않습니다.
general_dataset = load_dataset("json", data_files={
    "test": str(material_dir / "data" / "본실습" / "general_eval.jsonl"),
})
general_corpus_dataset = load_dataset("json", data_files={
    "corpus": str(material_dir / "data" / "본실습" / "general_corpus.jsonl"),
})
display(labor_dataset, labor_corpus_dataset, general_dataset, general_corpus_dataset)

### 3-2. push_to_hub

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">기존 부록의 <code style="font-size:0.95em;color:#345473;">push_to_hub</code>에 <code style="font-size:0.95em;color:#345473;">config_name</code>만 더합니다. 저장소가 없으면 만들고, 각 config의 데이터를 Parquet으로 올리며 열·split 정보를 카드에 기록합니다. <a href="https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.DatasetDict.push_to_hub">공식 DatasetDict 업로드 방법</a>을 따릅니다.</p></div>

In [ ]:
# 네 용도의 데이터를 같은 저장소의 별도 config로 올려 구분해 재사용합니다.
for config_name, dataset in [
    ("training-pairs", labor_dataset),
    ("corpus", labor_corpus_dataset),
    ("general-evaluation", general_dataset),
    ("general-corpus", general_corpus_dataset),
]:
    data_commit = dataset.push_to_hub(
        dataset_repo_id,
        config_name=config_name,  # 각 묶음의 열 구조와 split을 따로 보존합니다.
        private=False,  # 비공개로 올리려면 True. 아래 모델 저장소도 함께 바꿉니다.
        token=hf_token,
        commit_message=f"노무 임베딩 데이터: {config_name}",
    )
    print(config_name, "| 커밋 주소:", data_commit.commit_url)

### 3-3. 데이터셋 카드

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">데이터셋 카드는 저장소 첫 화면의 <code style="font-size:0.95em;color:#345473;">README.md</code>입니다. day71처럼 <code style="font-size:0.95em;color:#345473;">push_to_hub</code>가 만든 카드를 불러와 목적·출처·규모·한계를 채우고 다시 올립니다. 자동으로 기록된 네 config와 split 메타데이터는 유지합니다.</p></div>

In [ ]:
from huggingface_hub import DatasetCard
import json

# 자동 생성된 데이터 구조를 보존하고 출처·평가 방법을 카드에 덧붙입니다.
dataset_card = DatasetCard.load(dataset_repo_id, token=hf_token)
dataset_card.data.language = ["ko"]
dataset_card.data.task_categories = ["sentence-similarity"]

# 일반 평가의 원자료·선정 방식·한계도 다른 환경에서 확인할 수 있게 카드에 보존합니다.
general_source = json.loads((material_dir / "data" / "본실습" / "GENERAL_SOURCE.json").read_text(encoding="utf-8"))

# 규모는 파일에서 읽은 값으로 채우고, 원자료의 프리픽스를 바꾸지 않았음을 밝힙니다.
dataset_card.text = f"""
# 노무 조문 임베딩 데이터

## 목적과 출처
노무 상담 질의로 관련 법령 조문을 찾는 임베딩 모델의 학습·평가 데이터입니다.
원천은 국가법령정보센터 조문, 기존 노무 상담 질문과 생성·증강한 질의입니다.
원본 조문과 기존 상담 질문을 바탕으로 질의를 생성·증강하고 학습용과 평가용으로 나눴습니다.

## 구성과 사용
- 검색 문서: {len(labor_corpus_dataset["corpus"]):,}행. doc_id, law, topic, text.
- 학습 쌍: {len(labor_dataset["train"]):,}행. anchor, positive, doc_id.
- 검증 쌍: {len(labor_dataset["validation"]):,}행. anchor, positive, doc_id.
- 일반 평가 쌍: {len(general_dataset["test"]):,}행. anchor, positive, doc_id. 학습에는 사용하지 않습니다.
- 일반 검색 문서: {len(general_corpus_dataset["corpus"]):,}행. doc_id, domain, title, text.
- 학습/검증은 같은 세 열이며, 검색 평가는 검증의 질문(anchor)과 정답 ID(doc_id)를 사용합니다.
- 노무 검증 쌍은 검증 손실 확인과 Recall@5 최적 체크포인트 선택에도 사용하며 독립 테스트 결과를 뜻하지 않습니다.
- 원문에 query: 또는 passage:를 추가하지 않았습니다.

## 일반 검색 평가의 출처와 방법
관광 388문서/43질의, 상식 30문서/30질의, 수업용 가상 도서관 15문서/10질의입니다.
관광은 한국관광공사 TourAPI(공공누리 제1유형)를 사용한 기존 day58 자료이며, 상식은 day70의 question/reference_answer를 검색 쌍으로 바꿨습니다.
도서관은 day61의 수업용 가상 규정이며 정답 문서 하나인 질문을 선택했습니다.
정답 doc_id로 문서의 domain을 찾아 분야별 후보 문서에서 검색하고, 전체는 83질의의 순위를 합산합니다.
433개 문서를 섞은 단일 검색으로 평가하면 교안과 조건이 달라집니다.

원자료 경로·해시와 변환 기록:
```json
{json.dumps(general_source, ensure_ascii=False, indent=2)}
```

## 한계
검증 쌍은 질의마다 정답 조문 하나를 기록합니다. 다른 관련 조문도 있을 수 있습니다.
질문과 정답 연결에 오류가 남을 수 있으며 법률 자문을 위한 자료가 아닙니다.
일반 평가는 소형 검색 표본입니다. 짧은 상식 문서의 천장 효과와 가상 도서관 자료의 한계를 고려하며 일반능력 전체를 대표하지 않습니다.
"""
card_commit = dataset_card.push_to_hub(dataset_repo_id, token=hf_token, commit_message="데이터셋 카드 작성")
print("커밋 주소:", card_commit.commit_url)

## 4. 임베딩 모델 전체 올리기

### 4-1. 모델 폴더와 카드

`labor_tuned`는 교안 01의 `save_pretrained`가 만든 폴더입니다. 가중치만 따로 올리지 않고 아래 파일을 포함한 폴더 전체를 사용합니다.

| 파일 | 역할 |
|---|---|
| `model.safetensors`, `config.json` | 학습한 가중치와 모델 구조 |
| `tokenizer.json`, `tokenizer_config.json` | 문장을 토큰으로 바꾸는 방법 |
| `modules.json`, `config_sentence_transformers.json` | 임베딩 모델을 구성하는 모듈과 설정 |
| `1_Pooling/config.json` | 토큰 벡터를 문장 벡터로 모으는 방법 |
| `README.md` | 모델의 출처·학습 데이터·사용법을 적는 모델 카드 |

In [ ]:
from huggingface_hub import ModelCard

# 자동 생성 카드의 모델 정보를 유지하고, 데이터셋 링크와 사용 조건을 채웁니다.
model_card = ModelCard.load(model_dir / "README.md")
model_card.data.base_model = "intfloat/multilingual-e5-small"
model_card.data.datasets = [dataset_repo_id]
model_card.data.language = ["ko"]
model_card.data.license = "mit"  # 베이스 모델의 MIT 라이선스를 따릅니다.
model_card.data.library_name = "sentence-transformers"
model_card.data.pipeline_tag = "sentence-similarity"
model_card.text = f"""
# 노무 조문 검색용 임베딩 모델

## 모델과 학습 데이터
intfloat/multilingual-e5-small을 노무 질의와 정답 조문의 쌍으로 학습한 전체 임베딩 모델입니다.
학습 데이터 저장소: {dataset_repo_id}
모델 저장소: {model_repo_id}
같은 데이터셋 저장소의 general-evaluation/general-corpus는 다른 분야 검색의 퇴행 점검용이며 가중치 학습에는 사용하지 않았습니다.

## 사용 방법
SentenceTransformer("{model_repo_id}")로 모델 전체를 불러옵니다.
질의에 query: 를, 문서에 passage: 를 한 번씩 붙이고 normalize_embeddings=True로 인코딩합니다.
입력 상한은 특수 토큰과 프리픽스를 포함한 512토큰이며 임베딩은 384차원입니다.

## 한계
검색 점수는 평가 데이터에 따라 달라집니다. 다른 분야와 새 조문은 별도로 평가해야 합니다.
유사도가 높은 문서가 법률적 정답임을 보장하지 않으며 법률 자문용 모델이 아닙니다.
"""
# 보완한 카드를 모델 폴더에 저장해 다음 단계에서 가중치와 함께 공유합니다.
model_card.save(model_dir / "README.md")
print(model_card.data.to_yaml())

### 4-2. 모델 저장소에 업로드

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">모델 저장소의 종류는 <code style="font-size:0.95em;color:#345473;">model</code>입니다. <code style="font-size:0.95em;color:#345473;">upload_folder</code>는 <code style="font-size:0.95em;color:#345473;">save_pretrained</code>가 저장한 토크나이저와 Pooling 설정까지 함께 올립니다. 모델을 다시 학습하거나 어댑터를 병합할 단계는 없습니다.</p></div>

In [ ]:
# 원격 모델 저장소를 준비하고, 교안 01의 모델 구성 파일 전체를 올립니다.
api.create_repo(model_repo_id, private=False, exist_ok=True)  # 비공개로 올리려면 True
model_commit = api.upload_folder(
    repo_id=model_repo_id,
    folder_path=model_dir,  # 토크나이저와 Pooling 설정도 포함합니다.
    commit_message="노무 조문 검색용 임베딩 모델 전체",
)
print("모델 커밋 주소:", model_commit.commit_url)

# 가중치뿐 아니라 modules.json, 토크나이저, Pooling 폴더도 있는지 봅니다.
model_files = api.list_repo_files(model_repo_id)
display(model_files)

### 4-3. 허브 화면 확인

- 누를 곳: 프로필 사진 → 내 계정 → Datasets 또는 Models → 위에서 만든 저장소
- 주소: 아래 셀에서 내 계정에 맞는 주소를 확인합니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">데이터셋 페이지에서는 <code style="font-size:0.95em;color:#345473;">training-pairs</code>의 <code style="font-size:0.95em;color:#345473;">train</code>·<code style="font-size:0.95em;color:#345473;">validation</code>, <code style="font-size:0.95em;color:#345473;">corpus</code>, <code style="font-size:0.95em;color:#345473;">general-evaluation</code>의 <code style="font-size:0.95em;color:#345473;">test</code>, <code style="font-size:0.95em;color:#345473;">general-corpus</code>를 선택해 용도와 열을 확인합니다. 모델 페이지에서는 모델 카드의 데이터셋 링크와 Files and versions의 전체 구성 파일을 확인합니다.</p></div>

In [ ]:
# 계정 이름을 직접 바꿔 적지 않아도 내 저장소의 정확한 주소를 확인할 수 있습니다.
print("데이터셋:", f"https://huggingface.co/datasets/{dataset_repo_id}")
print("데이터셋 파일:", f"https://huggingface.co/datasets/{dataset_repo_id}/tree/main")
print("모델:", f"https://huggingface.co/{model_repo_id}")
print("모델 파일:", f"https://huggingface.co/{model_repo_id}/tree/main")

## 5. 허브에서 받아 검색하기

### 5-1. 용도별 데이터 받기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">다른 파드에서도 토큰·계정·저장소 이름을 준비한 뒤 이 절을 실행합니다. 노무 학습·검증·검색 문서와 일반 평가 쌍·문서를 같은 저장소에서 받습니다. 공개 저장소는 토큰 없이 받을 수 있고, 여기서는 비공개에도 같은 코드로 접근하도록 <code style="font-size:0.95em;color:#345473;">token=hf_token</code>을 넘깁니다.</p></div>

In [ ]:
import pandas as pd

# 다른 파드에서도 노무 학습·검증·검색 문서를 용도에 맞게 다시 받습니다.
hub_dataset = load_dataset(dataset_repo_id, "training-pairs", token=hf_token)
hub_corpus = load_dataset(dataset_repo_id, "corpus", split="corpus", token=hf_token)
hub_train = hub_dataset["train"]
hub_eval = hub_dataset["validation"]
# 일반 퇴행 평가에는 질문·정답 쌍과 분야를 구분할 검색 문서가 모두 필요합니다.
hub_general_eval = load_dataset(dataset_repo_id, "general-evaluation", split="test", token=hf_token)
hub_general_corpus = load_dataset(dataset_repo_id, "general-corpus", split="corpus", token=hf_token)

# 다시 받은 자료의 행 수와 열을 확인해 공유할 때의 구성과 대조합니다.
hub_data_table = pd.DataFrame([
    {"config": "training-pairs", "split": "train", "행 수": len(hub_train), "열": hub_train.column_names},
    {"config": "training-pairs", "split": "validation", "행 수": len(hub_eval), "열": hub_eval.column_names},
    {"config": "corpus", "split": "corpus", "행 수": len(hub_corpus), "열": hub_corpus.column_names},
    {"config": "general-evaluation", "split": "test", "행 수": len(hub_general_eval), "열": hub_general_eval.column_names},
    {"config": "general-corpus", "split": "corpus", "행 수": len(hub_general_corpus), "열": hub_general_corpus.column_names},
])
display(hub_data_table)

### 5-2. 모델 전체 받기

`SentenceTransformer`에 로컬 폴더 대신 저장소 이름을 줍니다. 모델의 가중치·토크나이저·Pooling 설정을 함께 읽습니다. [SentenceTransformer 모델 로딩](https://sbert.net/docs/package_reference/sentence_transformer/model.html)을 사용합니다.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# 기존 교안과 같은 장치 선택입니다. RunPod에서는 cuda를 사용합니다.
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("계산 장치:", device)

# e5 계열의 질의·문서 프리픽스는 교안 01과 같습니다.
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "
pd.set_option("display.max_colwidth", None)

# 공개·비공개 모두 같은 토큰 인자로 받습니다. 별도 베이스 모델이나 어댑터 객체는 만들지 않습니다.
hub_model = SentenceTransformer(model_repo_id, token=hf_token, device=device)

# 모델이 제공하는 입력 상한과 벡터 크기를 확인합니다.
hub_model_table = pd.DataFrame({
    "저장소": [model_repo_id],
    "입력 상한(토큰)": [hub_model.max_seq_length],
    "임베딩 차원": [hub_model.get_embedding_dimension()],
})
display(hub_model_table)

### 5-3. 같은 모델로 질의와 문서 인코딩

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01의 <code style="font-size:0.95em;color:#345473;">encode_texts</code>를 그대로 씁니다. 허브에서 받은 원문에도 질의·문서 프리픽스를 한 번씩 붙이고 정규화합니다. 이전 모델로 만든 문서 벡터와 섞지 않고, 받은 모델로 문서 벡터를 만듭니다.</p></div>

In [ ]:
def encode_texts(model, texts, prefix):
    """텍스트 목록에 프리픽스를 붙여 정규화된 임베딩(numpy 배열)으로 만듭니다."""
    return model.encode(
        [prefix + text for text in texts],   # 프리픽스를 붙인 문장 목록
        normalize_embeddings=True,           # 길이 1로 정규화해 벡터끼리의 내적이 곧 코사인 유사도가 되게 합니다
        show_progress_bar=False,             # 진행 막대를 끕니다(평가 때마다 여러 번 불러 출력이 길어지지 않게)
    )

In [ ]:
import numpy as np

# 교안 01과 같은 새 질의로, 허브에서 받은 모델과 문서를 연결합니다.
new_query = "그만둔 지 3주가 지났는데 회사가 아직 퇴직금을 안 줬어요. 이렇게 늦게 줘도 되나요?"
doc_ids = [doc["doc_id"] for doc in hub_corpus]
hub_doc_emb = encode_texts(hub_model, [doc["text"] for doc in hub_corpus], PASSAGE_PREFIX)
hub_query_emb = encode_texts(hub_model, [new_query], QUERY_PREFIX)
search_sims = hub_query_emb @ hub_doc_emb.T

# 정규화 벡터의 내적이 높은 문서부터 다섯 개를 보입니다.
search_rows = []
for rank, j in enumerate(np.argsort(-search_sims[0])[:5], start=1):
    search_rows.append({
        "순위": rank,
        "조문": doc_ids[j],
        "유사도": search_sims[0, j],
        "조문 내용": hub_corpus[int(j)]["text"],
    })
search_table = pd.DataFrame(search_rows)
display(search_table.round(3))

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li>모델은 저장소 이름으로, 문서는 데이터셋 저장소에서 받아 검색했습니다. 다른 파드에서도 같은 저장소를 사용해 학습 결과를 이어 쓸 수 있습니다.</li>
<li>질문은 퇴직금의 지급 시점을 묻습니다. 지급 시점을 다루는 조문과 시효를 다루는 조문을 구분해 읽습니다. Mac 로컬 재로딩 예에서는 지급 시점 조문이 2위였으므로, 1위의 유사도가 높다고 질문의 답으로 선택하지 않습니다. 실행 환경에 따라 순위는 달라질 수 있습니다.</li>
</ul></div>

## 6. 핵심 정리

- **분리해 공유:** 데이터는 기존 부록처럼 `push_to_hub`로, 모델은 `labor_tuned` 전체를 `upload_folder`로 올립니다.
- **데이터 구분:** 노무 학습·검증·검색 문서와 일반 퇴행 평가 쌍·검색 문서를 네 config로 공유합니다. 일반 평가는 학습에 넣지 않습니다.
- **같은 입력 형식:** 다시 받은 모델에도 `query: `와 `passage: `를 한 번씩 붙여 정규화합니다.
- **공개 범위:** `.env`의 쓰기 토큰으로 내 계정을 확인하고, 데이터와 모델의 `private` 설정을 함께 확인합니다.